# ABB Motor Predictive Maintenance — Exploratory Data Analysis

Bu notebook, modelleme öncesinde veriyi mühendislik ve veri analitiği açısından anlamak için hazırlanmıştır.

## Ana sorular
- Veri güvenilir mi?
- Motor hangi çalışma modlarında çalışıyor?
- 50 Hz ve 57 Hz davranışları nasıl ayrışıyor?
- Hangi titreşim yönü daha baskın?
- Güç, sıcaklık, hız, titreşim ve ivme arasında nasıl ilişkiler var?
- SQL ile üretilen trend ve baseline özellikleri mantıklı mı?
- `Future_Event` etiketi hangi sensör davranışlarıyla birlikte görülüyor?

> Çalışma gerçek mekanik arıza teşhisi yapmaz; normal davranıştan sapmaları ve erken uyarı göstergelerini inceler.


## 1. Kütüphaneler ve proje ayarları

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

print(
    "Credential bulundu:",
    bool(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))
)
print(
    "Credential yolu:",
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import GCP_PROJECT_ID

CLEAN_TABLE = f"{GCP_PROJECT_ID}.pm_clean.motor_measurements"
FEATURE_TABLE = f"{GCP_PROJECT_ID}.pm_features.motor_condition_features"
TRAINING_TABLE = f"{GCP_PROJECT_ID}.pm_features.training_dataset"

FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Project:", GCP_PROJECT_ID)
print("Figure output:", FIGURE_DIR)


Project: predictive-maintenance-504421
Figure output: c:\Users\Alihan\workintech-predictive-maintenance\reports\figures


## 2. BigQuery tablolarını yükleme

In [2]:
client = bigquery.Client(project=GCP_PROJECT_ID)

def read_bq_table(table_name: str) -> pd.DataFrame:
    query = f'''
    SELECT *
    FROM `{table_name}`
    ORDER BY Timestamp
    '''
    result = client.query(query).to_dataframe()
    result["Timestamp"] = pd.to_datetime(result["Timestamp"], utc=True, errors="coerce")
    return result

df_clean = read_bq_table(CLEAN_TABLE)
df_features = read_bq_table(FEATURE_TABLE)
df_training = read_bq_table(TRAINING_TABLE)

print("Clean:", df_clean.shape)
print("Features:", df_features.shape)
print("Training:", df_training.shape)


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

## 3. Genel veri özeti

In [ ]:
overview = pd.DataFrame({
    "Metric": [
        "Clean rows",
        "Feature rows",
        "Training rows",
        "First timestamp",
        "Last timestamp",
        "Running rows",
        "Stopped rows",
        "Operating modes",
    ],
    "Value": [
        len(df_clean),
        len(df_features),
        len(df_training),
        df_clean["Timestamp"].min(),
        df_clean["Timestamp"].max(),
        int(df_clean["Is_Running"].sum()),
        int((~df_clean["Is_Running"]).sum()),
        int(df_clean["Operating_Mode"].nunique(dropna=True)),
    ],
})
display(overview)


In [ ]:
display(
    df_clean[
        [
            "Timestamp", "Asset_ID", "Operating_Mode",
            "Speed_rpm", "Frequency_Hz", "Output_Power_kW",
            "Skin_Temp_C", "Overall_Vib_mm_s",
            "Data_Quality_Flag",
        ]
    ].head(10)
)


## 4. Veri kalitesi

In [ ]:
quality_summary = (
    df_clean["Data_Quality_Flag"]
    .value_counts(dropna=False)
    .rename_axis("Data_Quality_Flag")
    .reset_index(name="Row_Count")
)
quality_summary["Percentage"] = 100 * quality_summary["Row_Count"] / len(df_clean)
display(quality_summary)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(quality_summary["Data_Quality_Flag"].astype(str), quality_summary["Row_Count"])
plt.title("Data Quality Flags")
plt.xlabel("Quality flag")
plt.ylabel("Row count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_data_quality_flags.png", dpi=150)
plt.show()


In [ ]:
sensor_columns = [
    "Speed_rpm", "Frequency_Hz", "Output_Power_kW", "Skin_Temp_C",
    "Overall_Vib_mm_s", "Vib_Radial_mm_s", "Vib_Tangential_mm_s",
    "Vib_Axial_mm_s", "Acc_RMS_Axial_g", "Acc_RMS_Tangential_g",
    "Acc_RMS_Radial_g", "Pk_Pk_Tangential_g",
]

missing_summary = pd.DataFrame({
    "Column": sensor_columns,
    "Missing_Count": [int(df_clean[c].isna().sum()) for c in sensor_columns],
})
missing_summary["Missing_Percentage"] = 100 * missing_summary["Missing_Count"] / len(df_clean)
missing_summary = missing_summary.sort_values("Missing_Percentage", ascending=False)
display(missing_summary)


In [ ]:
plt.figure(figsize=(11, 6))
plt.barh(missing_summary["Column"], missing_summary["Missing_Percentage"])
plt.title("Missing Value Percentage")
plt.xlabel("Missing percentage")
plt.ylabel("Column")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_missing_values.png", dpi=150)
plt.show()


## 5. Ölçüm sıklığı ve zaman boşlukları

In [ ]:
time_gaps = (
    df_clean["Timestamp"].sort_values().diff()
    .dt.total_seconds().div(3600).dropna()
)
display(time_gaps.describe().to_frame("Hours_Between_Measurements"))


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(time_gaps.clip(upper=time_gaps.quantile(0.99)), bins=30)
plt.title("Time Gaps Between Measurements")
plt.xlabel("Hours")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_measurement_time_gaps.png", dpi=150)
plt.show()


## 6. Motor açık/kapalı davranışı

In [ ]:
state_summary = pd.DataFrame({
    "State": ["Running", "Stopped"],
    "Row_Count": [
        int(df_clean["Is_Running"].sum()),
        int((~df_clean["Is_Running"]).sum()),
    ],
})
state_summary["Percentage"] = 100 * state_summary["Row_Count"] / len(df_clean)
display(state_summary)


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(state_summary["State"], state_summary["Row_Count"])
plt.title("Running vs Stopped Measurements")
plt.xlabel("Motor state")
plt.ylabel("Row count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_running_vs_stopped.png", dpi=150)
plt.show()


In [ ]:
daily_runtime = (
    df_clean.assign(Date=df_clean["Timestamp"].dt.date)
    .groupby("Date", as_index=False)
    .agg(
        Total_Measurements=("Timestamp", "size"),
        Running_Measurements=("Is_Running", "sum"),
    )
)
daily_runtime["Running_Ratio"] = (
    daily_runtime["Running_Measurements"] / daily_runtime["Total_Measurements"]
)

plt.figure(figsize=(13, 5))
plt.plot(pd.to_datetime(daily_runtime["Date"]), daily_runtime["Running_Ratio"])
plt.title("Daily Running Measurement Ratio")
plt.xlabel("Date")
plt.ylabel("Running ratio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "05_daily_running_ratio.png", dpi=150)
plt.show()


## 7. Çalışma modları

In [ ]:
mode_summary = (
    df_clean["Operating_Mode"]
    .value_counts(dropna=False)
    .rename_axis("Operating_Mode")
    .reset_index(name="Row_Count")
)
mode_summary["Percentage"] = 100 * mode_summary["Row_Count"] / len(df_clean)
display(mode_summary)


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(mode_summary["Operating_Mode"].astype(str), mode_summary["Row_Count"])
plt.title("Operating Mode Distribution")
plt.xlabel("Operating mode")
plt.ylabel("Row count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "06_operating_mode_distribution.png", dpi=150)
plt.show()


## 8. Hız–frekans fiziksel tutarlılığı

In [ ]:
running = df_clean[
    df_clean["Is_Running"]
    & df_clean["Speed_rpm"].notna()
    & df_clean["Frequency_Hz"].notna()
].copy()

slope, intercept = np.polyfit(running["Frequency_Hz"], running["Speed_rpm"], 1)
predicted_speed = slope * running["Frequency_Hz"] + intercept

ss_res = np.sum((running["Speed_rpm"] - predicted_speed) ** 2)
ss_tot = np.sum((running["Speed_rpm"] - running["Speed_rpm"].mean()) ** 2)
r_squared = 1 - ss_res / ss_tot

print(f"Speed ≈ {slope:.3f} × Frequency + {intercept:.3f}")
print(f"R² = {r_squared:.4f}")


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(running["Frequency_Hz"], running["Speed_rpm"], alpha=0.5)

x_line = np.linspace(
    running["Frequency_Hz"].min(),
    running["Frequency_Hz"].max(),
    200,
)
plt.plot(x_line, slope * x_line + intercept, linestyle="--")

plt.title("Motor Speed vs Supply Frequency")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Speed (rpm)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "07_speed_vs_frequency.png", dpi=150)
plt.show()


## 9. Zaman serileri

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Frequency_Hz"])
plt.title("Frequency Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Frequency (Hz)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "08_frequency_over_time.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Overall_Vib_mm_s"])
plt.title("Overall Vibration Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Overall vibration (mm/s RMS)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "09_overall_vibration_time.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Skin_Temp_C"])
plt.title("Skin Temperature Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "10_temperature_over_time.png", dpi=150)
plt.show()


## 10. Çalışma moduna göre titreşim

In [ ]:
mode_vibration_summary = (
    df_clean[df_clean["Is_Running"]]
    .groupby("Operating_Mode")
    .agg(
        Row_Count=("Timestamp", "size"),
        Median_Overall=("Overall_Vib_mm_s", "median"),
        Median_Radial=("Vib_Radial_mm_s", "median"),
        Median_Tangential=("Vib_Tangential_mm_s", "median"),
        Median_Axial=("Vib_Axial_mm_s", "median"),
        P95_Overall=("Overall_Vib_mm_s", lambda s: s.quantile(0.95)),
    )
    .sort_values("Row_Count", ascending=False)
)
display(mode_vibration_summary)


In [ ]:
groups = list(df_clean[df_clean["Is_Running"]].groupby("Operating_Mode"))
box_data = [group["Overall_Vib_mm_s"].dropna().values for _, group in groups]
box_labels = [str(mode) for mode, _ in groups]

plt.figure(figsize=(10, 6))
plt.boxplot(box_data, labels=box_labels, showfliers=True)
plt.title("Overall Vibration by Operating Mode")
plt.xlabel("Operating mode")
plt.ylabel("Overall vibration (mm/s RMS)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "11_overall_vibration_by_mode.png", dpi=150)
plt.show()


## 11. Yönsel titreşim ve baskın eksen

In [ ]:
directional_data = df_clean[df_clean["Is_Running"]][
    ["Vib_Radial_mm_s", "Vib_Tangential_mm_s", "Vib_Axial_mm_s"]
].copy()

display(directional_data.describe().T)


In [ ]:
plt.figure(figsize=(9, 6))
plt.boxplot(
    [
        directional_data["Vib_Radial_mm_s"].dropna(),
        directional_data["Vib_Tangential_mm_s"].dropna(),
        directional_data["Vib_Axial_mm_s"].dropna(),
    ],
    labels=["Radial", "Tangential", "Axial"],
    showfliers=True,
)
plt.title("Directional Vibration Comparison")
plt.xlabel("Direction")
plt.ylabel("Vibration (mm/s RMS)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "12_directional_vibration.png", dpi=150)
plt.show()


In [ ]:
direction_map = {
    "Vib_Radial_mm_s": "Radial",
    "Vib_Tangential_mm_s": "Tangential",
    "Vib_Axial_mm_s": "Axial",
}

dominant_axis = directional_data.idxmax(axis=1).map(direction_map)
dominant_summary = (
    dominant_axis.value_counts()
    .rename_axis("Dominant_Axis")
    .reset_index(name="Row_Count")
)
display(dominant_summary)

plt.figure(figsize=(7, 5))
plt.bar(dominant_summary["Dominant_Axis"], dominant_summary["Row_Count"])
plt.title("Dominant Vibration Direction")
plt.xlabel("Direction")
plt.ylabel("Row count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "13_dominant_axis.png", dpi=150)
plt.show()


## 12. Tangential titreşim, ivme ve peak-to-peak

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Vib_Tangential_mm_s"])
plt.title("Tangential Vibration Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Tangential vibration (mm/s RMS)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "14_tangential_vibration_time.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Acc_RMS_Tangential_g"])
plt.title("Tangential Acceleration RMS Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Acceleration (g RMS)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "15_tangential_acceleration_time.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Pk_Pk_Tangential_g"])
plt.title("Tangential Peak-to-Peak Acceleration Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Peak-to-peak acceleration (g)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "16_peak_to_peak_time.png", dpi=150)
plt.show()


## 13. Güç, sıcaklık ve titreşim ilişkileri

In [ ]:
relationship_data = df_clean[
    df_clean["Is_Running"]
    & df_clean["Output_Power_kW"].notna()
    & df_clean["Skin_Temp_C"].notna()
].copy()

plt.figure(figsize=(9, 6))
plt.scatter(
    relationship_data["Output_Power_kW"],
    relationship_data["Skin_Temp_C"],
    alpha=0.5,
)
plt.title("Output Power vs Skin Temperature")
plt.xlabel("Output power (kW)")
plt.ylabel("Temperature (°C)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "17_power_vs_temperature.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    relationship_data["Output_Power_kW"],
    relationship_data["Overall_Vib_mm_s"],
    alpha=0.5,
)
plt.title("Output Power vs Overall Vibration")
plt.xlabel("Output power (kW)")
plt.ylabel("Overall vibration (mm/s RMS)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "18_power_vs_vibration.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
plt.boxplot(
    [
        df_clean.loc[df_clean["Is_Running"], "Skin_Temp_C"].dropna(),
        df_clean.loc[~df_clean["Is_Running"], "Skin_Temp_C"].dropna(),
    ],
    labels=["Running", "Stopped"],
    showfliers=True,
)
plt.title("Skin Temperature by Motor State")
plt.xlabel("Motor state")
plt.ylabel("Temperature (°C)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "19_temperature_by_state.png", dpi=150)
plt.show()


## 14. Spearman korelasyon matrisi

In [ ]:
correlation_columns = [
    "Speed_rpm", "Frequency_Hz", "Output_Power_kW", "Skin_Temp_C",
    "Overall_Vib_mm_s", "Vib_Radial_mm_s", "Vib_Tangential_mm_s",
    "Vib_Axial_mm_s", "Acc_RMS_Axial_g", "Acc_RMS_Tangential_g",
    "Acc_RMS_Radial_g", "Pk_Pk_Tangential_g",
]

corr = df_clean[df_clean["Is_Running"]][correlation_columns].corr(method="spearman")
display(corr.round(3))


In [ ]:
plt.figure(figsize=(12, 10))
image = plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(image, label="Spearman correlation")
plt.xticks(range(len(correlation_columns)), correlation_columns, rotation=90)
plt.yticks(range(len(correlation_columns)), correlation_columns)
plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "20_spearman_correlation.png", dpi=150)
plt.show()


## 15. SQL feature'larının doğrulanması

In [ ]:
feature_columns = [
    "Timestamp", "Operating_Mode", "Vib_Tangential_mm_s",
    "Change_Vib_Tangential", "Avg_3_Vib_Tangential",
    "Max_3_Vib_Tangential", "Deviation_Vib_Tangential",
    "Ratio_Vib_Tangential_To_Normal", "Is_High_Vib_Tangential",
]
display(df_features[feature_columns].head(15))


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(
    df_features["Timestamp"],
    df_features["Vib_Tangential_mm_s"],
    label="Current",
)
plt.plot(
    df_features["Timestamp"],
    df_features["Avg_3_Vib_Tangential"],
    label="Last 3 active measurements average",
)
plt.title("Current vs Rolling Tangential Vibration")
plt.xlabel("Timestamp")
plt.ylabel("Tangential vibration (mm/s RMS)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "21_current_vs_rolling.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_features["Deviation_Vib_Tangential"].dropna(), bins=40)
plt.axvline(0, linestyle="--")
plt.title("Tangential Vibration Deviation from Mode Baseline")
plt.xlabel("Deviation from operating-mode median")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "22_baseline_deviation.png", dpi=150)
plt.show()


## 16. Future Event etiketi

In [ ]:
event_summary = (
    df_training["Future_Event"]
    .value_counts()
    .rename_axis("Future_Event")
    .reset_index(name="Row_Count")
)
event_summary["Percentage"] = 100 * event_summary["Row_Count"] / len(df_training)
display(event_summary)

plt.figure(figsize=(7, 5))
plt.bar(event_summary["Future_Event"].astype(str), event_summary["Row_Count"])
plt.title("Future Event Class Distribution")
plt.xlabel("Future event label")
plt.ylabel("Row count")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "23_future_event_distribution.png", dpi=150)
plt.show()


In [ ]:
event_rate_by_mode = (
    df_training.groupby("Operating_Mode", as_index=False)
    .agg(
        Row_Count=("Future_Event", "size"),
        Event_Count=("Future_Event", "sum"),
        Event_Rate=("Future_Event", "mean"),
    )
    .sort_values("Event_Rate", ascending=False)
)
display(event_rate_by_mode)

plt.figure(figsize=(9, 5))
plt.bar(
    event_rate_by_mode["Operating_Mode"],
    100 * event_rate_by_mode["Event_Rate"],
)
plt.title("Future Event Rate by Operating Mode")
plt.xlabel("Operating mode")
plt.ylabel("Event rate (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "24_event_rate_by_mode.png", dpi=150)
plt.show()


## 17. Event olan ve olmayan kayıtların karşılaştırılması

In [ ]:
comparison_features = [
    "Vib_Tangential_mm_s",
    "Acc_RMS_Tangential_g",
    "Pk_Pk_Tangential_g",
    "Change_Vib_Tangential",
    "Avg_3_Vib_Tangential",
    "Deviation_Vib_Tangential",
    "Ratio_Vib_Tangential_To_Normal",
]

event_feature_summary = (
    df_training.groupby("Future_Event")[comparison_features]
    .median().T
    .rename(columns={0: "No_Future_Event", 1: "Future_Event"})
)

event_feature_summary["Difference"] = (
    event_feature_summary["Future_Event"]
    - event_feature_summary["No_Future_Event"]
)

display(event_feature_summary.sort_values("Difference", ascending=False))


In [ ]:
plt.figure(figsize=(8, 6))
plt.boxplot(
    [
        df_training.loc[
            df_training["Future_Event"] == 0,
            "Deviation_Vib_Tangential",
        ].dropna(),
        df_training.loc[
            df_training["Future_Event"] == 1,
            "Deviation_Vib_Tangential",
        ].dropna(),
    ],
    labels=["No Future Event", "Future Event"],
    showfliers=True,
)
plt.title("Tangential Baseline Deviation by Future Event")
plt.xlabel("Target class")
plt.ylabel("Tangential vibration deviation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "25_deviation_by_event.png", dpi=150)
plt.show()


## 18. En yüksek titreşim olayları

In [ ]:
event_columns = [
    "Timestamp", "Operating_Mode", "Overall_Vib_mm_s",
    "Vib_Radial_mm_s", "Vib_Tangential_mm_s", "Vib_Axial_mm_s",
    "Acc_RMS_Tangential_g", "Pk_Pk_Tangential_g",
    "Skin_Temp_C", "Output_Power_kW",
]

top_events = (
    df_clean[event_columns]
    .sort_values("Overall_Vib_mm_s", ascending=False)
    .head(20)
)
display(top_events)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean["Timestamp"], df_clean["Overall_Vib_mm_s"])
plt.scatter(
    top_events["Timestamp"],
    top_events["Overall_Vib_mm_s"],
    label="Top vibration events",
)
plt.title("Overall Vibration and Top Events")
plt.xlabel("Timestamp")
plt.ylabel("Overall vibration (mm/s RMS)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "26_top_vibration_events.png", dpi=150)
plt.show()


## 19. Otomatik EDA özeti

In [ ]:
most_common_mode = (
    df_clean.loc[df_clean["Is_Running"], "Operating_Mode"]
    .value_counts().idxmax()
)

dominant_direction = (
    dominant_summary.sort_values("Row_Count", ascending=False)
    .iloc[0]["Dominant_Axis"]
)

summary_findings = pd.DataFrame({
    "Finding": [
        "Most common running mode",
        "Dominant vibration direction",
        "Future event rate",
        "Missing power rate",
        "Speed-frequency R²",
        "Maximum overall vibration",
        "Median running temperature",
        "Median stopped temperature",
    ],
    "Value": [
        most_common_mode,
        dominant_direction,
        f"{100 * df_training['Future_Event'].mean():.2f}%",
        f"{100 * df_clean['Output_Power_kW'].isna().mean():.2f}%",
        f"{r_squared:.4f}",
        f"{df_clean['Overall_Vib_mm_s'].max():.3f} mm/s",
        f"{df_clean.loc[df_clean['Is_Running'], 'Skin_Temp_C'].median():.2f} °C",
        f"{df_clean.loc[~df_clean['Is_Running'], 'Skin_Temp_C'].median():.2f} °C",
    ],
})
display(summary_findings)


## 20. EDA'dan modele geçiş

Model notebook'una geçmeden önce şu sonuçlar kontrol edilmelidir:

- `Speed_rpm` ve `Frequency_Hz` çok benzer bilgi taşıyorsa aynı modelde ikisini birlikte kullanmak gereksiz olabilir.
- `Overall_Vib_mm_s`, yönsel titreşimlerin maksimumundan türetiliyorsa yönsel değişkenlerle birlikte kullanıldığında tekrar bilgi oluşturabilir.
- 50 Hz ve 57 Hz çalışma modları aynı normal titreşim seviyesine sahip değildir.
- Tekil spike ile kalıcı yükseliş ayrılmalıdır.
- `Future_Event` sınıf dengesizliği modelde ele alınmalıdır.
- Model başarısı accuracy yerine recall, precision, F1 ve yanlış alarm sayısıyla değerlendirilmelidir.

Notebook çalıştırıldığında grafikler `reports/figures/` klasörüne PNG olarak kaydedilir.
